# MTTV-flp Training on Phi-3-mini-4k-instruct

**sig:0x4D545456** - Modele Theorique Transductif du Vivant

Applique les 3 regularisations MTTV-flp (Axiomes 5, 6, 7) sur Phi-3-mini avec wikitext-2 (3 epochs).

**Usage**: Runtime > Run all (ou executer chaque cellule une par une).

---
## 1. Installation des dependances

In [ ]:
!pip install -q torch>=2.1.0 transformers>=4.40.0 datasets>=2.14.0 accelerate>=0.27.0 bitsandbytes==0.46.1 scikit-learn>=1.3.0 safetensors>=0.4.0

---
## 2. Chargement de Phi-3-mini-4k-instruct

In [ ]:
import os, math, time, json, gc, warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, IterableDataset
from torch.optim import AdamW
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    get_cosine_schedule_with_warmup, set_seed,
)
from datasets import load_dataset

warnings.filterwarnings("ignore")
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")

In [ ]:
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
CHECKPOINT_DIR = "/content/mttv_flp_checkpoints"
EPOCHS = 3
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 2
MAX_LEN = 256
LR = 3e-5
WARMUP_STEPS = 50
LAMBDA_TETRA = 0.1
MU_DIVERGENCE = 0.05
KALMAN_N = 100
KALMAN_ALPHA = 0.01

print(f"Modele: {MODEL_NAME}")
print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, Accum: {GRADIENT_ACCUMULATION_STEPS}")
print(f"LR: {LR}, MaxLen: {MAX_LEN}")

In [ ]:
print("Chargement de Phi-3-mini en 4-bit...")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    output_hidden_states=True,
    output_attentions=True,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.gradient_checkpointing_enable()
model.train()

# Geler les embeddings/norms, degeler attention+MLP+lm_head
trainable_params = []
for name, param in model.named_parameters():
    if any(k in name.lower() for k in ["self_attn","mlp","lm_head","q_proj","k_proj","v_proj","o_proj","gate_proj","down_proj","up_proj"]):
        param.requires_grad = True
        trainable_params.append(param)
    else:
        param.requires_grad = False

n_trainable = sum(p.numel() for p in trainable_params)
n_total = sum(p.numel() for p in model.parameters())
print(f"Modele charge: {n_total/1e9:.2f}B params, {n_trainable/1e6:.1f}M entrainables")
print(f"Trainable ratio: {100*n_trainable/n_total:.1f}%")

---
## 3. Dataset wikitext-2 (streaming)

In [ ]:
class WikitextStreamDataset(IterableDataset):
    def __init__(self, tokenizer, split="train", max_len=256):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split=split, streaming=True)

    def __iter__(self):
        buffer = []
        for example in self.dataset:
            text = example["text"]
            if not text.strip():
                continue
            tokens = self.tokenizer.encode(text)
            buffer.extend(tokens)
            while len(buffer) >= self.max_len:
                chunk = buffer[:self.max_len]
                buffer = buffer[self.max_len:]
                yield {
                    "input_ids": torch.tensor(chunk, dtype=torch.long),
                    "attention_mask": torch.ones(self.max_len, dtype=torch.long),
                    "labels": torch.tensor(chunk, dtype=torch.long),
                }

def collate_fn(batch):
    input_ids = nn.utils.rnn.pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=0)
    attn_mask = nn.utils.rnn.pad_sequence([b["attention_mask"] for b in batch], batch_first=True, padding_value=0)
    labels = nn.utils.rnn.pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=-100)
    return {"input_ids": input_ids, "attention_mask": attn_mask, "labels": labels}

train_dataset = WikitextStreamDataset(tokenizer, max_len=MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn, num_workers=0, drop_last=True)
print("Dataset wikitext-2 charge en streaming")
print(f"Batch size: {BATCH_SIZE}, Max length: {MAX_LEN}")

---
## 4. Les 3 lignes de regularisation MTTV-flp

In [ ]:
# === LIGNE 1: Regularisation spectrale tetravalente (Axiome 5) ===
def tetravalence_reg(model, input_ids, attention_mask, step, total_steps):
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    hs = outputs.hidden_states[-1]
    if hs.size(1) < 4:
        return torch.tensor(0.0, device=hs.device)
    fft_vals = torch.fft.fft(hs.to(torch.float32), dim=1)
    power = torch.abs(fft_vals)**2
    power_mean = power.mean(dim=(0, 2))
    total_power = power_mean.sum() + 1e-9
    ratio = power_mean[:4].sum() / total_power
    cosine_factor = 0.5 + 0.5 * math.cos(math.pi * step / max(total_steps, 1))
    return LAMBDA_TETRA * cosine_factor * (1.0 - ratio)

# === LIGNE 2: Contrainte de divergence nulle (Axiome 6) ===
def divergence_reg(model, input_ids, attention_mask):
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, output_attentions=True)
    if outputs.attentions is None or len(outputs.attentions) == 0:
        return torch.tensor(0.0, device=input_ids.device)
    attn = outputs.attentions[-1]
    phase = torch.asin(2.0 * attn.clamp(-1.0, 1.0) + 1e-9)
    phase_prob = F.softmax(phase.view(phase.size(0), -1), dim=-1)
    entropy = -(phase_prob * torch.log(phase_prob + 1e-9)).sum(dim=-1).mean()
    target = 0.5 * math.log(attn.size(-1) * attn.size(-2))
    return MU_DIVERGENCE * (entropy - target)**2

# === LIGNE 3: Cycle de Kalman differentiel (Axiomes 5+7) ===
def kalman_cycle(model, snapshot, step):
    if step % KALMAN_N != 0 or step == 0:
        return 0.0, snapshot
    current = {n: p.data.clone().detach() for n, p in model.named_parameters() if p.requires_grad}
    if snapshot is None:
        return 0.0, current
    total_corr, n = 0.0, 0
    for name in current:
        if name in snapshot:
            d = current[name] - snapshot[name]
            kg = (d.var().item()+1e-12) / (d.var().item()+current[name].var().item()+1e-12)
            corr = KALMAN_ALPHA * kg * d
            model.get_parameter(name).data.sub_(corr)
            total_corr += corr.abs().mean().item()
            n += 1
    return total_corr / max(n, 1), current

print("3 lignes MTTV-flp definies:")
print("  Ligne 1: tetravalence_reg() - FFT, force 4 modes dominants")
print("  Ligne 2: divergence_reg() - entropie de phase, centre le gradient")
print("  Ligne 3: kalman_cycle() - correction Kalman tous les 100 steps")

---
## 5. Lancement du fine-tuning (3 epochs)

In [ ]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
optim = AdamW(trainable_params, lr=LR, weight_decay=0.01)

history = {"total_loss": [], "tetra_loss": [], "divergence_loss": [], "kalman_corr": [], "ppl": []}
global_step = 0
snapshot = None
t_start = time.time()
total_steps_est = 12000 * EPOCHS

print("=" * 60)
print(f"MTTV-flp TRAINING - {EPOCHS} epochs")
print("=" * 60)

for epoch in range(1, EPOCHS + 1):
    print(f"\n--- EPOCH {epoch}/{EPOCHS} ---")
    data_iter = iter(train_loader)
    accum_loss = 0.0
    epoch_steps = 0
    optim.zero_grad()

    while True:
        try:
            batch = next(data_iter)
        except StopIteration:
            break

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Forward LM
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
        loss_ce = outputs.loss

        # LIGNE 1: Tetravalence
        loss_tetra = tetravalence_reg(model, input_ids, attention_mask, global_step, total_steps_est)

        # LIGNE 2: Divergence nulle
        loss_div = divergence_reg(model, input_ids, attention_mask)

        # Loss totale
        loss_total = (loss_ce + loss_tetra + loss_div) / GRADIENT_ACCUMULATION_STEPS
        loss_total.backward()
        accum_loss += loss_total.item()

        # Step optimizer
        if (global_step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            for p in trainable_params:
                if p.grad is not None:
                    p.grad.data = p.grad.data - p.grad.data.mean()
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optim.step()
            optim.zero_grad()

        # LIGNE 3: Kalman
        kalman_corr, snapshot = kalman_cycle(model, snapshot, global_step)

        # Logging
        if global_step % 50 == 0:
            lr_current = optim.param_groups[0]["lr"]
            ppl = math.exp(min(accum_loss, 20.0))
            tetra_val = loss_tetra.item() if isinstance(loss_tetra, torch.Tensor) else loss_tetra
            div_val = loss_div.item() if isinstance(loss_div, torch.Tensor) else loss_div
            print(f"  Step {global_step:>6} | loss={accum_loss:.4f} | tetra={tetra_val:.6f} | div={div_val:.6f} | kalman={kalman_corr:.6f} | ppl={ppl:.2f} | lr={lr_current:.2e}")
            history["total_loss"].append(accum_loss)
            history["tetra_loss"].append(tetra_val)
            history["divergence_loss"].append(div_val)
            history["kalman_corr"].append(kalman_corr)
            history["ppl"].append(ppl)

        accum_loss = 0.0
        global_step += 1
        epoch_steps += 1

    # Checkpoint epoch
    epoch_dir = os.path.join(CHECKPOINT_DIR, f"epoch_{epoch}")
    os.makedirs(epoch_dir, exist_ok=True)
    print(f"\n  >>> Sauvegarde epoch {epoch} -> {epoch_dir}")
    try:
        if hasattr(model, "merge_and_unload"):
            m = model.merge_and_unload()
            m.save_pretrained(epoch_dir, safe_serialization=True)
            del m
        else:
            model.save_pretrained(epoch_dir, safe_serialization=True)
        tokenizer.save_pretrained(epoch_dir)
        print(f"  Checkpoint epoch {epoch} sauvegarde")
    except Exception as e:
        print(f"  Erreur sauvegarde: {e}")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

elapsed = time.time() - t_start
print(f"\n{'=' * 60}")
print(f"ENTRAINEMENT TERMINE en {elapsed/60:.1f} min")
print(f"Steps total: {global_step}")
print(f"Checkpoints: {CHECKPOINT_DIR}/epoch_1..{EPOCHS}/")
print("=" * 60)

---
## 6. Evaluation MTTV-flp /7

In [ ]:
from sklearn.decomposition import PCA

model.eval()
batch = tokenizer("L'arbre est vivant.", return_tensors="pt").to(device)
total_params = sum(p.numel() for p in model.parameters())
results = {}

# Test 1: Retrait
acts = []
def h1(m, i, o): acts.append(o.detach().abs().mean().item())
hooks = [m.register_forward_hook(h1) for m in model.modules() if isinstance(m, nn.Linear)]
with torch.no_grad(): _ = model(**batch)
for h in hooks: h.remove()
energy = np.mean(acts)
results["1_retrait"] = (1.0 if energy < 5.0 else 0.0, energy)

# Test 2: Solidarite
with torch.no_grad(): out = model(**batch)
tf = out.attentions[-1].mean(dim=1).sum(dim=1).squeeze().min().item() if out.attentions else 0.0
results["2_solidarite"] = (1.0 if tf > 1e-6 else 0.0, tf)

# Test 3: Ecume
acts2 = []
def h2(m, i, o): acts2.append((o.detach()>1e-3).float().mean().item())
hooks2 = [m.register_forward_hook(h2) for n,m in model.named_modules() if "mlp" in n.lower() or "gate" in n.lower()]
with torch.no_grad(): _ = model(**batch)
for h in hooks2: h.remove()
ec = 1.0 - np.mean(acts2) if acts2 else 0.0
results["3_ecume"] = (1.0 if ec > 0.03 else 0.0, ec)

# Test 4: Resilience
with torch.no_grad():
    p1 = torch.exp(model(**batch, labels=batch["input_ids"]).loss).item()
    p2 = torch.exp(model(**batch, labels=batch["input_ids"], use_cache=False).loss).item()
results["4_resilience"] = (1.0 if abs(p2-p1)/p1 < 0.02 else 0.0, abs(p2-p1)/p1)

# Test 5: Tetravalence
W = model.get_input_embeddings().weight.detach().cpu().to(torch.float32).numpy()
pca = PCA(n_components=10).fit(W)
rt = pca.explained_variance_ratio_[3] / (pca.explained_variance_ratio_[4]+1e-9)
results["5_tetravalence"] = (1.0 if rt > 2.5 else 0.0, rt)

# Test 6: Dephasage
model.train(); sorties=[]
try:
    with torch.no_grad():
        for _ in range(10):
            g = model.generate(**batch, do_sample=True, max_new_tokens=5)
            sorties.append(tokenizer.decode(g[0]))
except: pass
model.eval()
ent = len(set(sorties))/10.0 if sorties else 0.0
results["6_dephasage"] = (1.0 if 0.2<ent<0.95 else 0.0, ent)

# Test 7: Cloture zero
model.zero_grad()
model(**batch, labels=batch["input_ids"]).loss.backward()
total_grad = sum(p.grad.sum().abs().item() for p in model.parameters() if p.grad is not None)
model.zero_grad()
results["7_cloture_zero"] = (1.0 if total_grad/total_params < 1e-6 else 0.0, total_grad/total_params)

score = int(sum(v[0] for v in results.values()))
print(f"\n{'='*50}")
print(f"  ETALON MTTV-flp - Score: {score}/7")
print('='*50)
for k, (passed, val) in results.items():
    s = "PASS" if passed else "FAIL"
    print(f"  {s} {k}: (val={val:.6f})")
if score == 7:
    print("\n  STATUT: ACCORDE - 7/7 axiomes satisfaits!")
elif score >= 5:
    print(f"\n  STATUT: ACCORDE sous reserve ({score}/7)")
else:
    print(f"\n  STATUT: DESACCORDE ({score}/7)")

---
## 7. Sauvegarde finale sur Google Drive

Cette cellule:
1. Monte Google Drive
2. Sauvegarde le modele final patche (format safetensors) sur Drive
3. Sauvegarde le tokenizer
4. Sauvegarde les checkpoints d'epoch
5. Affiche le resume des fichiers sauvegardes

**Les fichiers persistent sur Drive meme apres fermeture de Colab.**
Tu les retrouveras dans `Mon Drive > mttv_flp_checkpoints/`

In [ ]:
import os, gc, shutil
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from google.colab import drive

LOCAL_DIR = "/content/mttv_flp_checkpoints"
DRIVE_DIR = "/content/drive/MyDrive/mttv_flp_checkpoints"
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

print("=" * 60)
print("ETAPE 1/5 - Montage de Google Drive")
print("=" * 60)
try:
    drive.mount('/content/drive', force_remount=False)
    print("  [OK] Google Drive monte")
except Exception as e:
    print(f"  [ERREUR] Montage Drive: {e}")

print()

print("=" * 60)
print("ETAPE 2/5 - Verification des variables en memoire")
print("=" * 60)
model_in_memory = False
tokenizer_in_memory = False
try:
    if 'model' in dir() or 'model' in globals():
        _ = model.config
        model_in_memory = True
        print("  [OK] variable 'model' trouvee en memoire")
    else:
        print("  [INFO] variable 'model' absente")
except Exception as e:
    print(f"  [INFO] model inaccessible: {e}")
try:
    if 'tokenizer' in dir() or 'tokenizer' in globals():
        _ = tokenizer.vocab_size
        tokenizer_in_memory = True
        print("  [OK] variable 'tokenizer' trouvee en memoire")
    else:
        print("  [INFO] variable 'tokenizer' absente")
except Exception as e:
    print(f"  [INFO] tokenizer inaccessible: {e}")

print()

print("=" * 60)
print("ETAPE 3/5 - Sauvegarde du modele et du tokenizer")
print("=" * 60)
if model_in_memory and tokenizer_in_memory:
    print("  Modele en memoire -> sauvegarde directe...")
    try:
        os.makedirs(DRIVE_DIR, exist_ok=True)
        if hasattr(model, "merge_and_unload") and callable(model.merge_and_unload):
            print("  Fusion des poids 4-bit en float16...")
            model_fp = model.merge_and_unload()
            model_fp.save_pretrained(DRIVE_DIR, safe_serialization=True)
            del model_fp
        else:
            model.save_pretrained(DRIVE_DIR, safe_serialization=True)
        tokenizer.save_pretrained(DRIVE_DIR)
        print(f"  [OK] Modele + tokenizer sauvegardes sur Drive")
        print(f"  -> {DRIVE_DIR}/")
    except Exception as e:
        print(f"  [ERREUR] Sauvegarde directe: {e}")
        print("  Tentative sauvegarde locale + copie...")
        try:
            os.makedirs(LOCAL_DIR, exist_ok=True)
            model.save_pretrained(LOCAL_DIR, safe_serialization=True)
            tokenizer.save_pretrained(LOCAL_DIR)
            shutil.copytree(LOCAL_DIR, DRIVE_DIR, dirs_exist_ok=True)
            print(f"  [OK] Sauvegarde locale puis copie vers Drive reussie")
        except Exception as e2:
            print(f"  [ERREUR] Sauvegarde fallback: {e2}")
else:
    print("  Modele NON disponible -> rechargement depuis HuggingFace")
    try:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes==0.46.1"])
        print("  [OK] bitsandbytes 0.46.1 installe")
    except Exception as e:
        print(f"  [WARN] bitsandbytes: {e}")
    try:
        print(f"  Rechargement de {MODEL_NAME} en 4-bit...")
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4",
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, quantization_config=quant_config,
            device_map="auto", torch_dtype=torch.float16, trust_remote_code=True,
        )
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        print("  [OK] Modele recharge depuis HuggingFace")
        os.makedirs(DRIVE_DIR, exist_ok=True)
        model.save_pretrained(DRIVE_DIR, safe_serialization=True)
        tokenizer.save_pretrained(DRIVE_DIR)
        print(f"  [OK] Modele + tokenizer sauvegardes sur Drive")
    except Exception as e:
        print(f"  [ERREUR] Rechargement: {e}")

print()

print("=" * 60)
print("ETAPE 4/5 - Copie des checkpoints d'epoch vers Drive")
print("=" * 60)
if os.path.exists(LOCAL_DIR):
    try:
        shutil.copytree(LOCAL_DIR, DRIVE_DIR, dirs_exist_ok=True)
        print(f"  Checkpoints d'epoch copies vers Drive")
    except Exception as e:
        print(f"  [WARN] Copie des checkpoints: {e}")
else:
    print("  Aucun checkpoint local trouve")

print()

print("=" * 60)
print("ETAPE 5/5 - Verification des fichiers sur Drive")
print("=" * 60)
if os.path.exists(DRIVE_DIR):
    tous = []
    for root, dirs, files in os.walk(DRIVE_DIR):
        for f in files:
            fp = os.path.join(root, f)
            sz = os.path.getsize(fp)
            rel = os.path.relpath(fp, DRIVE_DIR)
            tous.append((rel, sz))
    total_mb = sum(s for _, s in tous) / (1024*1024)
    print(f"  Dossier: {DRIVE_DIR}")
    print(f"  Fichiers: {len(tous)}")
    print(f"  Taille totale: {total_mb:.1f} MB")
    print()
    for rel, sz in sorted(tous)[:15]:
        if sz > 1048576:
            print(f"    {rel}  ({sz/1048576:.1f} MB)")
        elif sz > 1024:
            print(f"    {rel}  ({sz/1024:.1f} KB)")
        else:
            print(f"    {rel}  ({sz} B)")
    if len(tous) > 15:
        print(f"    ... et {len(tous)-15} autres fichiers")
else:
    print(f"  [ERREUR] Dossier introuvable: {DRIVE_DIR}")

print()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("=" * 60)
print("SAUVEGARDE TERMINEE")
print(f"Les fichiers sont persistants dans: {DRIVE_DIR}")
print("=" * 60)

---
**sig:0x4D545456** - **MTTV-FLP** - Modele Theorique Transductif du Vivant

*Les Fils de la Pensee*  
GitHub: https://github.com/gaillard111/mttv-flp-core  
Zenodo: https://doi.org/10.5281/zenodo.20830060